In [1]:
# %pip install wbgapi
# %pip install yfinance
# %pip install matplotlib

In [2]:
import yfinance as yf
import matplotlib.pyplot as plt
import wbgapi as wb
import pandas as pd

In [3]:
# Get the data for the crude_oil
# Date is an index
crude_oil = yf.download('CL=F','2001-01-01','2021-01-01')
crude_oil.head()

[*********************100%***********************]  1 of 1 completed


,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
2001-01-02,27.250000,27.400000,26.600000,27.200001,27.200001,52321
2001-01-03,27.230000,28.139999,27.049999,27.950001,27.950001,66628
2001-01-04,28.200001,28.780001,27.850000,28.200001,28.200001,74383
2001-01-05,28.150000,28.799999,27.799999,28.000000,28.000000,63852
2001-01-08,28.200001,28.400000,27.150000,27.350000,27.350000,76058


In [4]:
#Create a new column from the index of the dataframe
crude_oil['Date'] = crude_oil.index
crude_oil['Year'] = pd.DatetimeIndex(crude_oil['Date']).year
crude_oil.groupby(crude_oil['Date'].map(lambda x: x.year))
crude_oil.set_index('Year', inplace=True)
crude_oil.head()

,Open,High,Low,Close,Adj Close,Volume,Date
Year,,,,,,,
2001,27.250000,27.400000,26.600000,27.200001,27.200001,52321,2001-01-02
2001,27.230000,28.139999,27.049999,27.950001,27.950001,66628,2001-01-03
2001,28.200001,28.780001,27.850000,28.200001,28.200001,74383,2001-01-04
2001,28.150000,28.799999,27.799999,28.000000,28.000000,63852,2001-01-05
2001,28.200001,28.400000,27.150000,27.350000,27.350000,76058,2001-01-08


In [5]:
crude_oil.describe()

,Open,High,Low,Close,Adj Close,Volume
count,5021.000000,5021.000000,5021.000000,5021.000000,5021.000000,5.021000e+03
mean,62.378821,63.309142,61.380520,62.375433,62.375433,3.012032e+05
std,25.830898,26.041726,25.594766,25.846645,25.846645,2.228863e+05
min,-14.000000,13.690000,-40.320000,-37.630001,-37.630001,0.000000e+00
25%,42.810001,43.700001,41.910000,42.869999,42.869999,1.120200e+05
50%,58.980000,59.730000,58.020000,58.900002,58.900002,2.561030e+05
75%,82.500000,83.760002,81.290001,82.660004,82.660004,3.968560e+05
max,145.190002,147.270004,143.220001,145.289993,145.289993,2.288230e+06


In [18]:
# crude_oil_cleaned = crude_oil.groupby("Year")["Adj Close"].mean().to_frame()
crude_oil_cleaned = crude_oil.groupby("Year").agg(avg_oil_price_in_usd=('Adj Close', 'mean'))
# crude_oil_cleaned = crude_oil.rename(columns = {'Unnamed: 1':'Mean_Oil_Price'})
crude_oil_cleaned.to_csv('crude_oil_cleaned.csv')
crude_oil_cleaned.head()


,avg_oil_price_in_usd
Year,
2001,25.960405
2002,26.150440
2003,30.994400
2004,41.469076
2005,56.704502


In [7]:
wb.series.info(q='gdp')

id,value
EG.GDP.PUSE.KO.PP,GDP per unit of energy use (PPP $ per kg of oil equivalent)
EG.GDP.PUSE.KO.PP.KD,GDP per unit of energy use (constant 2017 PPP $ per kg of oil equivalent)
EG.USE.COMM.GD.PP.KD,"Energy use (kg of oil equivalent) per $1,000 GDP (constant 2017 PPP)"
NY.GDP.DEFL.KD.ZG,"Inflation, GDP deflator (annual %)"
NY.GDP.DEFL.KD.ZG.AD,"Inflation, GDP deflator: linked series (annual %)"
NY.GDP.DEFL.ZS,GDP deflator (base year varies by country)
NY.GDP.DEFL.ZS.AD,GDP deflator: linked series (base year varies by country)
NY.GDP.DISC.CN,Discrepancy in expenditure estimate of GDP (current LCU)
NY.GDP.DISC.KN,Discrepancy in expenditure estimate of GDP (constant LCU)
NY.GDP.MKTP.CD,GDP (current US$)


In [8]:
GDP=wb.data.DataFrame('NY.GDP.MKTP.KD.ZG',
                               'USA', time=range(2001,2021))
cleaned_GDP = GDP.T
cleaned_GDP['Year'] = cleaned_GDP.index
cleaned_GDP['Year'] = cleaned_GDP['Year'].str[2:]
cleaned_GDP.set_index('Year', inplace=True)
cleaned_GDP.rename(columns = {'USA':'GDP_growth'}, inplace = True)
cleaned_GDP.to_csv('cleaned_GDP.csv')
cleaned_GDP.head()



economy,GDP_growth
Year,
2001,0.998341
2002,1.741695
2003,2.861211
2004,3.798891
2005,3.513214


In [9]:
wb.series.info(q='interest')

id,value
FR.INR.DPST,Deposit interest rate (%)
FR.INR.LEND,Lending interest rate (%)
FR.INR.LNDP,"Interest rate spread (lending rate minus deposit rate, %)"
FR.INR.RINR,Real interest rate (%)
GC.XPN.INTP.CN,Interest payments (current LCU)
GC.XPN.INTP.RV.ZS,Interest payments (% of revenue)
GC.XPN.INTP.ZS,Interest payments (% of expense)
,7 elements


In [10]:
interest_rate=wb.data.DataFrame('FR.INR.LEND',
                               'USA', time=range(2001,2021))
cleaned_interest_rate = interest_rate.T
cleaned_interest_rate['Year'] = cleaned_interest_rate.index
cleaned_interest_rate['Year'] = cleaned_interest_rate['Year'].str[2:]
cleaned_interest_rate.set_index('Year', inplace=True)
cleaned_interest_rate.rename(columns = {'USA':'interest_rate'}, inplace = True)
cleaned_interest_rate.to_csv('cleaned_interest_rate.csv')
cleaned_interest_rate.tail()

economy,interest_rate
Year,
2016,3.511667
2017,4.096667
2018,4.904167
2019,5.282500
2020,3.544167


In [24]:
#To avoid NaN errors
cleaned_GDP.index = crude_oil_cleaned.index
cleaned_interest_rate.index = cleaned_GDP.index


In [25]:
gdp_interest_oil = pd.concat([cleaned_GDP, cleaned_interest_rate, crude_oil_cleaned], axis=1)
gdp_interest_oil.head()

,GDP_growth,interest_rate,avg_oil_price_in_usd
Year,,,
2001,0.998341,6.921667,25.960405
2002,1.741695,4.675000,26.150440
2003,2.861211,4.122500,30.994400
2004,3.798891,4.340000,41.469076
2005,3.513214,6.189167,56.704502
